# 08 — Reading PDFs: text layers, page routing and HunyuanOCR-1.5

After this notebook you can get text out of any PDF the way a production RAG pipeline does: read the text layer when there is a good one, fix two-column reading order, pull tables out as tables, detect the pages that need OCR, render them at the right resolution, send them to **HunyuanOCR-1.5** (Tencent's 1B OCR model, July 2026) through its OpenAI-compatible server, turn the markdown and HTML tables it returns into chunks, measure how wrong the OCR was, and explain with the paper's own numbers why its speculative decoding is 6.37× faster in one setting and 2.14× in another.

**Time:** ~30 min · **Runs:** offline on CPU in a few seconds. The OCR model itself needs a GPU server; without one, a stand-in answers so every cell still runs (set `HUNYUAN_OCR_URL` to use the real thing) · **Needs:** [chunking](01_chunking_strategies_and_semantic_chunking.ipynb), [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb) · **Course chapter:** GenAI Flow → *Reading PDFs: HunyuanOCR*

## Why this matters in interviews

- "How do you extract text from PDFs reliably?" is one of the most common RAG questions, and the honest answer is a **router**, not a library: text layer, table extractor, OCR, per page. You will build one.
- The new generation of OCR is a small vision-language model that writes markdown. Interviewers now ask *why it is slow* and *how it was made fast* — which is a question about speculative decoding and Amdahl's law, and you will answer it from measured numbers.
- The follow-ups are production ones: how many GPUs for a million pages, how do you know the OCR is wrong, what happens to page numbers and tables on the way into chunks.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt08` | How do you extract text from PDFs reliably? |
| `dt25` | What is HunyuanOCR-1.5, and why is it fast? |
| `dt26` | How do you decide, page by page, whether a PDF needs OCR? |
| `dt27` | Why does speculative decoding give OCR 6.37× in one setup and 2.14× in another? |
| `dt28` | How do you size an OCR fleet for a million pages? |
| `dt29` | How do you know the OCR output is wrong? |
| `dt30` | What breaks when OCR output flows into a RAG index? |

In [ ]:
import base64
import os
import re
import time
from html.parser import HTMLParser
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf                                  # pip install pymupdf  (imported as fitz in older code)
from PIL import Image, ImageFilter

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)
print("PyMuPDF", pymupdf.__version__)

## 1. A PDF where we know every answer

**In plain English:** to test an extractor you need a document whose correct text you already know. So we make one: four A4 pages that between them contain every problem real PDFs have.

| Page | What it is | The problem it carries |
|------|-----------|------------------------|
| 1 | Annual report, two columns | The PDF writes the lines **across** both columns, so the stored text order interleaves them |
| 2 | Invoice with a ruled table | A text extractor returns the cells as a list of words; the grid is gone |
| 3 | Signed contract page, **scanned** | No text layer at all: it is a picture |
| 4 | Cover letter | Nothing wrong. It should never go near an OCR model |

The ground truth is kept in Python variables, so every later cell can check itself.

In [ ]:
A4_W, A4_H = pymupdf.paper_size("a4")          # 595 x 842 points (1 pt = 1/72 inch)

LEFT = ["Acme Cloud grew revenue 41% in 2026,", "driven by analytics seats and the", "new usage-based pricing tier.",
        "Gross margin reached 78%, up from", "74%, as storage costs fell after", "the move to tiered object storage."]
RIGHT = ["Edge Gateway shipments doubled to", "18,400 units. The EG-300 now ships", "from Rotterdam and Dallas, and the",
         "rugged outdoor model launches in", "the first quarter of 2027 at a", "list price of 649 dollars."]
INVOICE = pd.DataFrame({"Item": ["Edge Gateway EG-300", "Express delivery", "Total"],
                        "Qty": ["4", "1", ""],
                        "Price": ["1,196.00", "25.00", "1,221.00"]})
CONTRACT = ["SERVICE AGREEMENT", "This agreement is made between Acme Cloud B.V. and Northwind Retail Ltd.",
            "Monthly fee: 1,221.00 USD, payable within 30 days of the invoice date.",
            "Term: 24 months from 1 October 2026. Either party may terminate with 90 days notice.",
            "Signed for Acme Cloud: J. de Vries      Signed for Northwind: A. Okafor"]
LETTER = ["Dear Ms Okafor,", "Thank you for choosing Acme Cloud. Your gateways leave our Rotterdam",
          "warehouse this week, and a tracking link will follow by email.", "Kind regards, Jan de Vries"]


def two_column_page(doc):
    page = doc.new_page(width=A4_W, height=A4_H)
    page.insert_text((50, 60), "Annual Report 2026", fontsize=16, fontname="hebo")
    for i, (l, r) in enumerate(zip(LEFT, RIGHT)):   # line by line ACROSS the page, as many generators do
        y = 100 + 16 * i
        page.insert_text((50, y), l, fontsize=10, fontname="helv")
        page.insert_text((310, y), r, fontsize=10, fontname="helv")


def table_page(doc):
    page = doc.new_page(width=A4_W, height=A4_H)
    page.insert_text((50, 60), "Invoice 2026-0917", fontsize=16, fontname="hebo")
    xs, y0, h = [50, 280, 360, 520], 90, 24
    rows = [list(INVOICE.columns)] + INVOICE.values.tolist()
    for i, row in enumerate(rows):
        for j, cell in enumerate(row):
            page.insert_text((xs[j] + 6, y0 + i * h + 16), cell, fontsize=10, fontname="hebo" if i == 0 else "helv")
    for i in range(len(rows) + 1):                   # the ruled grid a table extractor looks for
        page.draw_line((xs[0], y0 + i * h), (xs[-1], y0 + i * h), color=(0, 0, 0), width=0.8)
    for x in xs:
        page.draw_line((x, y0), (x, y0 + len(rows) * h), color=(0, 0, 0), width=0.8)


def scanned_page(doc):
    # typeset the page, photograph it at 150 dpi, tilt and blur it slightly, and keep ONLY the picture
    tmp = pymupdf.open()
    p = tmp.new_page(width=A4_W, height=A4_H)
    for i, line in enumerate(CONTRACT):
        p.insert_text((50, 70 + 22 * i), line, fontsize=14 if i == 0 else 10, fontname="hebo" if i == 0 else "helv")
    img = Image.open(BytesIO(p.get_pixmap(dpi=150).tobytes("png"))).convert("L")
    img = img.rotate(0.6, fillcolor=255, resample=Image.BICUBIC).filter(ImageFilter.GaussianBlur(0.6))
    buf = BytesIO(); img.save(buf, format="PNG")
    doc.new_page(width=A4_W, height=A4_H).insert_image(pymupdf.Rect(0, 0, A4_W, A4_H), stream=buf.getvalue())


def letter_page(doc):
    page = doc.new_page(width=A4_W, height=A4_H)
    for i, line in enumerate(LETTER):
        page.insert_text((50, 70 + 18 * i), line, fontsize=11, fontname="helv")


doc = pymupdf.open()
for make in (two_column_page, table_page, scanned_page, letter_page):
    make(doc)
PDF_BYTES = doc.tobytes()
doc = pymupdf.open(stream=PDF_BYTES, filetype="pdf")   # re-open, exactly as a file from disk
print(f"{doc.page_count} pages, {len(PDF_BYTES) / 1024:.0f} KB")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4.6))
titles = ["1. two columns", "2. ruled table", "3. scanned", "4. letter"]
for ax, page, t in zip(axes, doc, titles):
    pix = page.get_pixmap(dpi=40)
    ax.imshow(Image.open(BytesIO(pix.tobytes("png"))), cmap="gray")
    ax.set_title(t); ax.axis("off")
plt.suptitle("The test PDF, rendered", y=1.0)
plt.tight_layout(); plt.show()

## 2. The text layer: free, exact, and sometimes missing

**In plain English:** a PDF typed in Word or exported from a website stores the actual characters, with their positions. Reading them back is not OCR, it is just reading — milliseconds per page on a CPU, and no chance of misreading a 1 as a 7. A scanned page stores only a picture, so there is nothing to read.

**Technically:** `page.get_text()` walks the page's content stream and returns the characters it draws. It is exact for born-digital PDFs and returns nothing (or an old OCR layer) for scans.

In [ ]:
t0 = time.perf_counter()
texts = [page.get_text() for page in doc]
per_page_ms = (time.perf_counter() - t0) / doc.page_count * 1000

for i, t in enumerate(texts, 1):
    print(f"page {i}: {len(t):4d} characters | {t[:70].strip()!r}")
print(f"\n{per_page_ms:.2f} ms per page on this CPU")

assert len(texts[2].strip()) == 0, "the scanned page must have no text layer"
assert all(len(texts[i]) > 100 for i in (0, 1, 3)), "the born-digital pages must have real text"
assert per_page_ms < 200, "text-layer extraction should be milliseconds, not seconds"

## 3. Reading order: the two-column trap

**In plain English:** the text layer stores characters in the order the PDF *drew* them, not the order a person reads them. Page 1 was drawn line by line straight across the page, so a naive read interleaves the two columns into nonsense — and that nonsense is what gets embedded.

**Technically:** `sort=True` sorts blocks top-to-bottom then left-to-right, which does **not** fix columns: two lines at the same height still interleave. You have to decide which column each word belongs to first. `get_text("words")` gives every word with its box; assign a column by its x position, then sort within each column.

In [ ]:
page1 = doc[0]
naive = page1.get_text()
sorted_blocks = page1.get_text(sort=True)
print("NAIVE:\n", naive[:260], "\n")


def column_order(page, n_cols=2):
    # words -> (column, line y, x). Column = which slice of the page width the word starts in.
    words = page.get_text("words")              # (x0, y0, x1, y1, word, block, line, word_no)
    col_w = page.rect.width / n_cols
    keyed = sorted(words, key=lambda w: (int(w[0] // col_w), round(w[3]), w[0]))
    cols = {}
    for w in keyed:
        cols.setdefault(int(w[0] // col_w), []).append(w[4])
    return [" ".join(cols[c]) for c in sorted(cols)]


cols = column_order(page1)
print("COLUMN-AWARE:\n", cols[0][:120], "...\n", cols[1][:120], "...")

truth_left, truth_right = " ".join(LEFT), " ".join(RIGHT)
body = lambda s: " ".join(s.split())
assert truth_left not in body(naive) and truth_left not in body(sorted_blocks), \
    "the naive and sorted reads must interleave the columns - that is the trap"
assert cols[0].endswith(truth_left) and cols[1] == truth_right, "column-aware order must recover both columns exactly"

The title sits in the left slice, so it leads column 0 — which is where a reader would put it too. Real layouts are messier (three columns, a full-width figure caption), which is why layout-aware tools exist; but *knowing* that the text layer's order is drawing order is the insight interviewers look for.

## 4. Tables: text layer versus a table extractor

**In plain English:** a table read as text becomes a list of words, and "4" no longer knows it is the quantity of "Edge Gateway EG-300". For a born-digital table with ruled lines, a table extractor that follows the lines gets the grid back exactly — no OCR needed.

In [ ]:
page2 = doc[1]
print("AS TEXT:", " | ".join(page2.get_text().split("\n")[:10]))

tables = page2.find_tables()                     # follows the ruling lines
print(f"\nfound {len(tables.tables)} table(s)")
got = tables.tables[0].to_pandas().fillna("")
got.columns = [str(c) for c in got.columns]
print(got.to_string(index=False))

assert got.values.tolist() == INVOICE.values.tolist() and list(got.columns) == list(INVOICE.columns), \
    "the ruled table must come back cell for cell"

That works because the table was **born digital and ruled**. A borderless table, a table inside a scan, or one that spans a page break defeats it — those are the pages an OCR model that emits HTML tables is for.

## 5. The page router

**In plain English:** decide *per page*, because one contract often mixes typed pages with a scanned signature page. Use the text layer when it is present and clean; send a page to OCR when it is a picture, when its text layer is junk (an old OCR pass, or formulas flattened into symbols), or when it holds a table the text layer will destroy.

These are the same four rules as the router in the course chapter.

In [ ]:
JUNK_WORD = re.compile(r"[^\w.,;:'\"()$%&/+\-]|[A-Za-z]\d[A-Za-z]|\d[A-Za-z]\d")


def page_stats(page):
    text = page.get_text()
    words = text.split()
    area = page.rect.width * page.rect.height
    img_area = sum(pymupdf.Rect(i["bbox"]).get_area() for i in page.get_image_info())
    return {"chars": len(text.strip()),
            "img": min(1.0, img_area / area),
            "junk": sum(bool(JUNK_WORD.search(w)) for w in words) / max(1, len(words)),
            "tables": len(page.find_tables().tables)}


def route(s):
    if s["chars"] < 50:
        return "ocr", "no text layer worth the name: the page is a picture"
    if s["junk"] > 0.15:
        return "ocr", f"{s['junk']:.0%} of the text layer is junk"
    if s["img"] > 0.9:
        return "ocr", "mostly image under a thin text layer: an earlier OCR pass"
    if s["tables"]:
        return "ocr", f"{s['tables']} table(s): send to OCR, or to a table extractor if ruled"
    return "native", "clean text layer"


rows = []
for i, page in enumerate(doc, 1):
    s = page_stats(page)
    r, why = route(s)
    rows.append({"page": i, **s, "route": r, "why": why})
ROUTES = pd.DataFrame(rows)
print(ROUTES.round(2).to_string(index=False))

assert ROUTES.route.tolist() == ["native", "ocr", "ocr", "native"]
assert ROUTES.loc[2, "img"] > 0.99 and ROUTES.loc[0, "junk"] == 0

## 6. Rendering a page for the model

**In plain English:** OCR models read pictures, never the PDF itself, so your code renders each routed page to PNG. The resolution is a cost dial: too low and small print blurs, too high and the image costs more to encode on every page. 200 dpi is the usual default and 300 dpi is for footnotes and dense tables. HunyuanOCR-1.5 accepts images up to 4K: an A4 page at 300 dpi is about 3,500 pixels tall and fits; at 400 dpi it does not.

**Technically:** pixels = points × dpi / 72, per side. Pixel count grows with dpi squared.

In [ ]:
page3 = doc[2]
rows = []
for dpi in (72, 150, 200, 300, 400):
    pix = page3.get_pixmap(dpi=dpi)
    rows.append({"dpi": dpi, "width": pix.width, "height": pix.height,
                 "megapixels": pix.width * pix.height / 1e6})
RENDER = pd.DataFrame(rows)
print(RENDER.round(2).to_string(index=False))

w200 = RENDER.set_index("dpi").loc[200]
assert abs(w200.width - A4_W * 200 / 72) <= 1 and abs(w200.height - A4_H * 200 / 72) <= 1
assert RENDER.set_index("dpi").loc[300].height < 3840 < RENDER.set_index("dpi").loc[400].height,     "A4 fits the ~4K limit at 300 dpi and overshoots it at 400"
ratio = RENDER.set_index("dpi").loc[300].megapixels / w200.megapixels
assert abs(ratio - (300 / 200) ** 2) < 0.02, "pixels grow with dpi squared"
print(f"\n300 dpi costs {ratio:.2f}x the pixels of 200 dpi")

## 7. Calling HunyuanOCR-1.5

**In plain English:** the model runs behind a vLLM server that speaks the OpenAI API. You send the page image and the task prompt; you get markdown back. Because it is the OpenAI API, the client below is the same code for a GPU server or for llama.cpp on a laptop.

To start the real server (Linux, CUDA 13, Python 3.12 — from the [HunyuanOCR repo](https://github.com/Tencent-Hunyuan/HunyuanOCR)):

```bash
uv pip install "vllm>=0.25.1"
huggingface-cli download tencent/HunyuanOCR --local-dir ./HunyuanOCR --exclude "v1.0/*"
MODEL_PATH=./HunyuanOCR GPU=0 PORT=8000 bash inference/vLLM/serve.sh
```

then set `HUNYUAN_OCR_URL=http://<host>:8000/v1` before running this notebook. Without it, an **offline stand-in** returns what the model would return for our test pages — including one deliberate misread, so section 9 has something to catch. The stand-in is a fake; the request it receives is real.

The same thing in plain transformers, if you have a GPU but no server:

```python
from transformers import AutoProcessor, HunYuanVLForConditionalGeneration
processor = AutoProcessor.from_pretrained("tencent/HunyuanOCR", trust_remote_code=True)
model = HunYuanVLForConditionalGeneration.from_pretrained("tencent/HunyuanOCR", torch_dtype="bfloat16", device_map="auto")
messages = [{"role": "user", "content": [{"type": "image", "image": pil_image}, {"type": "text", "text": DOC_PARSE}]}]
inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                       return_dict=True, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=8192, do_sample=False, repetition_penalty=1.08)
```

In [ ]:
# the default doc_parse prompt, verbatim from the model card. In English: "Extract all the information in the
# body of the document image as markdown. Ignore the header and footer. Express tables as HTML and formulas as
# LaTeX. Organise the result in reading order."
DOC_PARSE = ("提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，表格用html格式表达，"
             "文档中公式用latex格式表示，按照阅读顺序组织进行解析。")


def build_request(png: bytes, prompt: str = DOC_PARSE) -> dict:
    url = "data:image/png;base64," + base64.b64encode(png).decode()
    return {"model": os.getenv("HUNYUAN_OCR_MODEL", "tencent/HunyuanOCR"),
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": prompt}]}],
            "temperature": 0.0, "top_p": 1.0, "max_tokens": 32768,
            "extra_body": {"top_k": -1, "repetition_penalty": 1.08}}   # the repo's sampling config


# what the model returns for our two routed pages, written by hand. One number is misread on purpose.
_STAND_IN = {
    2: "# Invoice 2026-0917\n\n<table><tr><td>Item</td><td>Qty</td><td>Price</td></tr>"
       "<tr><td>Edge Gateway EG-300</td><td>4</td><td>1,196.00</td></tr>"
       "<tr><td>Express delivery</td><td>1</td><td>25.00</td></tr>"
       "<tr><td>Total</td><td></td><td>1,221.00</td></tr></table>",
    3: "# SERVICE AGREEMENT\n\n" + "\n\n".join(CONTRACT[1:]).replace("1,221.00 USD", "1,227.00 USD"),
}


def ocr_page(png: bytes, page_no: int) -> str:
    base = os.getenv("HUNYUAN_OCR_URL")
    if base:
        from openai import OpenAI
        req = build_request(png)
        r = OpenAI(base_url=base, api_key="EMPTY").chat.completions.create(**req)
        return r.choices[0].message.content
    return _STAND_IN[page_no]


OCR_BACKEND = "HunyuanOCR-1.5 at " + os.environ["HUNYUAN_OCR_URL"] if os.getenv("HUNYUAN_OCR_URL") else "offline stand-in"
print("OCR backend:", OCR_BACKEND)

req = build_request(doc[2].get_pixmap(dpi=200).tobytes("png"))
shown = req["messages"][0]["content"][0]["image_url"]["url"]
print("request image:", shown[:40] + f"... ({len(shown) / 1024:.0f} KB of base64)")
print("request keys:", list(req))

In [ ]:
OCR_OUT = {}
for p in ROUTES.query("route == 'ocr'").page:
    png = doc[p - 1].get_pixmap(dpi=200).tobytes("png")
    t0 = time.perf_counter()
    OCR_OUT[p] = ocr_page(png, p)
    print(f"--- page {p} ({time.perf_counter() - t0:.2f} s) ---\n{OCR_OUT[p][:300]}\n")

## 8. From markdown to chunks

**In plain English:** the model's output is not ready to embed. Tables arrive as HTML, which a character splitter will happily cut in half, leaving rows without their header. And the default prompt drops headers and footers — so the page number is gone unless *you* carry it.

**Technically:** parse each `<table>` into rows, then write every row as a sentence that repeats the column names ("Item: Express delivery; Qty: 1; Price: 25.00"). Each chunk carries the page number from your loop as metadata.

In [ ]:
class TableParser(HTMLParser):
    # <table> HTML -> list of rows (lists of cell strings). Handles th and td alike.
    def __init__(self):
        super().__init__(); self.rows, self._row, self._cell = [], None, None
    def handle_starttag(self, tag, attrs):
        if tag == "tr": self._row = []
        elif tag in ("td", "th"): self._cell = ""
    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._row is not None: self._row.append(self._cell.strip()); self._cell = None
        elif tag == "tr" and self._row is not None: self.rows.append(self._row); self._row = None
    def handle_data(self, data):
        if self._cell is not None: self._cell += data


def to_chunks(markdown: str, page_no: int, source: str = "test.pdf") -> list[dict]:
    chunks, pos = [], 0
    for m in re.finditer(r"<table>.*?</table>", markdown, flags=re.S):
        prose = markdown[pos:m.start()].strip()
        if prose:
            chunks.append({"text": prose, "kind": "prose"})
        tp = TableParser(); tp.feed(m.group(0))
        header, *body = tp.rows
        for row in body:                          # one chunk per row, each carrying the header
            chunks.append({"text": "; ".join(f"{h}: {v}" for h, v in zip(header, row) if v), "kind": "table row"})
        pos = m.end()
    if markdown[pos:].strip():
        chunks.append({"text": markdown[pos:].strip(), "kind": "prose"})
    return [{**c, "page": page_no, "source": source} for c in chunks]


CHUNKS = [c for p, md_text in OCR_OUT.items() for c in to_chunks(md_text, p)]
for c in CHUNKS:
    print(f"page {c['page']}  {c['kind']:9s}  {c['text'][:90]!r}")

rows = [c["text"] for c in CHUNKS if c["kind"] == "table row"]
assert rows[1] == "Item: Express delivery; Qty: 1; Price: 25.00", "every table row must carry its header"
assert all(c["page"] in (2, 3) for c in CHUNKS), "every chunk must know its page"

## 9. How wrong was the OCR?

**In plain English:** an OCR model does not say "I could not read this". On a blurry digit it writes a confident, plausible wrong digit. Two checks catch most of it: **character error rate** against a labelled sample (how much text is wrong overall), and a **numbers diff** — the digits that matter most (amounts, dates, IDs) compared one by one, because a 0.2% error rate can still be the one digit in the price.

**Technically:** CER = edit distance / reference length. Edit distance (Levenshtein) is the minimum number of single-character inserts, deletes and substitutions — a classic dynamic-programming table.

In [ ]:
def levenshtein(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


norm = lambda s: " ".join(re.sub(r"[#*]", " ", s).split())
truth3 = norm(" ".join(CONTRACT))
got3 = norm(OCR_OUT[3])
cer = levenshtein(got3, truth3) / len(truth3)
nums = lambda s: re.findall(r"\d[\d,.]*\d|\d", s)
diff = [(t, g) for t, g in zip(nums(truth3), nums(got3)) if t != g]

print(f"character error rate on page 3: {cer:.2%}")
print("numbers that differ:", diff)

assert levenshtein("kitten", "sitting") == 3 and levenshtein("", "abc") == 3
if OCR_BACKEND == "offline stand-in":
    assert diff == [("1,221.00", "1,227.00")], "the numbers diff must catch the planted misread"
    assert cer < 0.01, "one wrong digit is a tiny CER - which is exactly why CER alone is not enough"

A CER under 1% and a wrong monthly fee on a contract. That is the argument for checking numbers separately — and, when a page *has* a text layer, for comparing OCR numbers against it.

## 10. Why it is fast: speculative decoding, measured

**In plain English:** the model writes the page one token at a time — an average page is about 1,400 tokens. HunyuanOCR-1.5 adds a tiny draft model (90.7M parameters) that guesses a block of 16 tokens at once; the 1B model checks the whole block in one pass, keeps the part it agrees with, and writes its own token at the first mistake. The output is identical. The speedup depends on how many guesses survive, and on how much of the time was decoding in the first place.

**Technically:** if each drafted token is accepted with probability α and the block holds k tokens, the expected tokens per pass is (1 − α^(k+1)) / (1 − α) (Leviathan et al., 2023). The paper measures 8.4–9.5 tokens per pass on vLLM.

In [ ]:
def expected_accept(a, k):
    return k + 1 if a == 1 else (1 - a ** (k + 1)) / (1 - a)


rng = np.random.default_rng(0)
def simulate(a, k, passes=20_000):
    # each pass: draft k tokens, keep them until the first rejection, then the target adds one token of its own
    kept = [(np.cumprod(rng.random(k) < a).sum() + 1) for _ in range(passes)]
    return float(np.mean(kept))


alphas = np.linspace(0.5, 0.99, 50)
plt.figure(figsize=(7, 3.4))
plt.plot(alphas, [expected_accept(a, 16) for a in alphas], color="#0891b2", lw=2)
plt.axhspan(8.39, 9.50, color="#0891b2", alpha=0.12, label="measured on vLLM: 8.39-9.50 tokens per pass")
plt.xlabel("per-token acceptance probability a"); plt.ylabel("tokens kept per pass (k = 16)")
plt.legend(frameon=False); plt.title("How good must the draft be?"); plt.show()

for a in (0.8, 0.9, 0.92):
    print(f"a = {a}: formula {expected_accept(a, 16):.2f}, simulated {simulate(a, 16):.2f} tokens per pass")
assert abs(simulate(0.9, 16) - expected_accept(0.9, 16)) < 0.15, "the simulation must agree with the formula"
assert 8.3 < expected_accept(0.9, 16) < 8.4, "a draft that is right 90% of the time keeps ~8.3 tokens a pass"

So a draft that is right about 9 times in 10 already explains the measured acceptance. Then why is the measured speedup only 2.14×, not 9×? The paper times pages in five buckets by output length. Fit **time per page = fixed cost + tokens × cost per token** to each mode:

In [ ]:
# HunyuanOCR-1.5 paper, Table 4, vLLM rows: latency (s) and output tokens/s per length bucket
PAPER = pd.DataFrame({
    "bucket":  ["0-256", "256-512", "512-1024", "1024-2048", "2048+"],
    "ar_lat":  [0.950, 1.156, 1.926, 3.071, 6.660], "ar_tps": [217.9, 341.8, 395.9, 466.5, 514.3],
    "df_lat":  [0.723, 0.746, 1.086, 1.435, 2.901], "df_tps": [286.4, 529.5, 702.4, 998.6, 1183.0],
    "speedup": [1.31, 1.55, 1.77, 2.14, 2.30]})
PAPER["tokens"] = PAPER.ar_lat * PAPER.ar_tps              # tokens per page = seconds x tokens/s

ar_per_tok, ar_fixed = np.polyfit(PAPER.tokens, PAPER.ar_lat, 1)
df_per_tok, df_fixed = np.polyfit(PAPER.df_lat * PAPER.df_tps, PAPER.df_lat, 1)
print(f"autoregressive: {ar_fixed:.3f} s fixed + {ar_per_tok * 1000:.2f} ms per token")
print(f"DFlash:         {df_fixed:.3f} s fixed + {df_per_tok * 1000:.2f} ms per token")
print(f"decoding alone gets {ar_per_tok / df_per_tok:.2f}x faster; the fixed cost does not move")

assert abs(ar_fixed - df_fixed) < 0.01, "DFlash leaves the fixed cost (image encode + prefill) unchanged"
assert round(ar_fixed, 2) == 0.53 and round(ar_per_tok * 1000, 2) == 1.79 and round(df_per_tok * 1000, 2) == 0.69

N = np.linspace(0, 4000, 100)
plt.figure(figsize=(7, 3.8))
plt.plot(N, ar_fixed + ar_per_tok * N, color="#d97706", lw=2, label="autoregressive (fit)")
plt.plot(N, df_fixed + df_per_tok * N, color="#0891b2", lw=2, label="DFlash (fit)")
plt.scatter(PAPER.tokens, PAPER.ar_lat, color="#d97706", zorder=3)
plt.scatter(PAPER.df_lat * PAPER.df_tps, PAPER.df_lat, color="#0891b2", zorder=3)
plt.axhline(ar_fixed, color="gray", ls=":", label=f"fixed cost {ar_fixed:.2f} s")
plt.xlabel("output tokens per page"); plt.ylabel("seconds per page"); plt.legend(frameon=False)
plt.title("Five measured buckets, one straight line each"); plt.show()

That is Amdahl's law with real numbers. DFlash makes each token 2.6× cheaper, and nothing else: reading and encoding the image costs 0.53 s either way. A short page is mostly fixed cost, so it gains 1.31×; a long table gains 2.30×; no page on vLLM can ever pass 2.6×.

The **6.37×** in plain transformers is the same trick against a slower baseline: there each ordinary step pays per-step overhead (34.9 s a page, 41 tokens/s) that vLLM's engine has already removed, and DFlash pays it once per ~9 tokens instead of once per token.

In [ ]:
pred = (ar_fixed + ar_per_tok * PAPER.tokens) / (df_fixed + df_per_tok * PAPER.tokens)
print(pd.DataFrame({"bucket": PAPER.bucket, "tokens": PAPER.tokens.round(), "measured": PAPER.speedup,
                    "model": pred.round(2)}).to_string(index=False))
print(f"\nceiling for an infinitely long page: {ar_per_tok / df_per_tok:.2f}x")
print(f"transformers: {34.850 / 5.474:.2f}x  (34.850 s -> 5.474 s a page)")

assert np.all(np.diff(pred) > 0), "longer pages must gain more"
assert np.all(np.abs(pred - PAPER.speedup) < 0.2), "the straight-line model must reproduce every bucket's speedup"
assert round(34.850 / 5.474, 2) == 6.37

## 11. Sizing the fleet

**In plain English:** the single-request speedup is a latency number. To size hardware you need throughput with many pages in flight — and speculative decoding's lead shrinks under load, because batching already keeps the GPU busy and drafting starts competing for it.

| pages in flight | plain (pages/s) | DFlash (pages/s) | speedup |
|---|---|---|---|
| 1 | 0.330 | 0.706 | 2.14× |
| 4 | 0.676 | 1.529 | 2.26× |
| 32 | 1.098 | 1.980 | 1.80× |

*(HunyuanOCR-1.5 paper, Table 6, one accelerator; the paper does not name the GPU.)*

In [ ]:
def gpu_hours(pages, ocr_share, pages_per_s):
    return pages * ocr_share / pages_per_s / 3600


corpus, share = 1_000_000, (ROUTES.route == "ocr").mean()
for name, pps in [("plain, 32 in flight", 1.098), ("DFlash, 32 in flight", 1.980), ("DFlash, one at a time", 0.706)]:
    print(f"{name:24s}: {gpu_hours(corpus, 1.0, pps):6.0f} GPU-h for everything, "
          f"{gpu_hours(corpus, share, pps):6.0f} GPU-h routed ({share:.0%} of pages need OCR)")

gpus_24h = int(np.ceil(gpu_hours(corpus, share, 1.980) / 24))
print(f"\nto OCR the routed pages of a million-page corpus in 24 hours: {gpus_24h} GPUs")
assert round(gpu_hours(corpus, 1.0, 1.980)) == 140
assert gpu_hours(corpus, share, 1.980) < gpu_hours(corpus, 1.0, 1.980), "routing must save GPU time"

## Try it yourself

**Exercise 1.** A page with an old, bad OCR layer: add a scanned page that also carries an invisible text layer full of misread words (`render_mode=3` makes text invisible, as OCR tools do). Does the router send it back to OCR, and which rule catches it?

In [ ]:
# Solution — try it yourself first, then run this
extra = pymupdf.open()
extra.insert_pdf(pymupdf.open(stream=PDF_BYTES, filetype="pdf"), from_page=2, to_page=2)   # copy the scanned page
bad = extra[0]
for i, line in enumerate(["Serv1ce Agr~ement betw##n Acme Cl0ud B.V. and N0rthw1nd",
                          "M0nthly f#e: 1,22l.O0 USD pay~ble with1n 3O d#ys"]):
    bad.insert_text((50, 100 + 22 * i), line, fontsize=10, render_mode=3)
s = page_stats(bad)
print({k: round(v, 2) for k, v in s.items()}, "->", route(s))
assert route(s)[0] == "ocr" and s["chars"] >= 50, "the junk rule (or the image rule) must catch it despite the text layer"

**Exercise 2.** With the fitted latency model, how many output tokens must a page have before DFlash is at least 2× faster on vLLM?

In [ ]:
# Solution — try it yourself first, then run this
# ar_fixed + N * ar_per_tok = 2 * (df_fixed + N * df_per_tok)  ->  solve for N
n2 = (2 * df_fixed - ar_fixed) / (ar_per_tok - 2 * df_per_tok)
print(f"about {n2:,.0f} tokens: a full page of prose. Short pages never reach 2x.")
assert 1000 < n2 < 1500

**Exercise 3.** You must OCR 250,000 pages overnight (8 hours), and 40% of them need OCR. How many GPUs with DFlash at 32 in flight, and how many without it?

In [ ]:
# Solution — try it yourself first, then run this
need = lambda pps: int(np.ceil(gpu_hours(250_000, 0.40, pps) / 8))
print(f"DFlash: {need(1.980)} GPUs, plain decoding: {need(1.098)} GPUs")
assert need(1.980) < need(1.098)

## Say it in an interview

- **30 seconds:** "I route per page. If the text layer is present and clean I read it — milliseconds, exact — fixing reading order for columns and using a table extractor for ruled tables. Pages that are pictures, have a junk text layer, or hold tables I cannot extract go to an OCR model; today that is a small VLM like HunyuanOCR-1.5, which returns markdown with HTML tables and LaTeX. I render at 200 dpi, keep the page number as metadata because the model drops headers and footers, turn table rows into self-describing chunks, and check OCR numbers against the text layer or a labelled sample."
- **Numbers:** HunyuanOCR-1.5 is 1B parameters, 94.74 on OmniDocBench v1.6, 0.706 pages/s for one request and 1.98 pages/s at 32 in flight. DFlash keeps ~9 tokens a pass; 6.37× in transformers, 2.14× in vLLM, 1.80× at 32 concurrent. Fixed cost ≈ 0.53 s a page; per-token cost 1.79 → 0.69 ms.
- **Why not 9×:** Amdahl — speculative decoding only speeds up decoding, never the image encode and prefill; each pass costs ~3.5 ordinary steps; and under load there is no idle compute left to spend on drafting.
- **Traps:** OCR-ing born-digital pages (slower *and* less accurate); trusting `sort=True` to fix columns; splitting a `<table>` mid-row; trusting CER when one digit is wrong; sizing GPUs from the single-request speedup; forgetting the licence (Tencent Hunyuan Community License, not Apache).
- **Follow-ups:** "Two-stage or end-to-end?" (PaddleOCR-VL and GLM-OCR run a layout model then read regions in parallel — the fast default without speculation; end-to-end models cover more tasks and, with DFlash, are faster). "How would you evaluate?" (a few hundred labelled pages of *your* documents: CER, table cell accuracy, numbers diff).

## Next

- [Chunking strategies](01_chunking_strategies_and_semantic_chunking.ipynb) — what happens to the markdown next.
- [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb) — put ingestion, retrieval and generation together.
- [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) — the decoding costs that speculative decoding attacks.